# Notebook 06: Color Following

## Before You Start
> **If anything behaves unexpectedly, restart the kernel first: Kernel menu → Restart Kernel and Clear All Outputs. Then run the cells from the top.**

## ADAS Connection
This notebook brings together everything you have learned so far. The robot will now **see a color and drive toward it** -- the same fundamental loop used in real autonomous vehicles:

1. **Perceive** -- camera captures a frame
2. **Detect** -- find the target in the frame
3. **Decide** -- which way should we go?
4. **Act** -- send commands to the motors

This perception-decision-action loop runs dozens of times per second in a real self-driving car. Tesla calls it their **autonomy stack**. You are about to build a miniature version of it.

---

## How It Works
Remember from Notebook 05 that the detected color blob has an **X position** in the frame. The frame is 640 pixels wide so the center is at X=320.

- If X < 320 -- the target is to the **left** -- turn left
- If X > 320 -- the target is to the **right** -- turn right
- If X is close to 320 -- the target is **centered** -- drive forward
- If nothing detected -- **stop**

The **dead zone** is a range around center where the robot drives straight instead of turning. Without it the robot would constantly overcorrect left and right trying to perfectly center the target.

```
|----LEFT----|--DEAD ZONE--|----RIGHT----|
0           270    320    370           640
```

---

## The Code
Run this cell first to set up the camera and motors.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
import threading
import time
import sys
from IPython.display import display

sys.path.insert(0, '/home/pi/lab')
import motors

def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

# Open camera
try:
    cap.release()
except:
    pass

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
cap.set(cv2.CAP_PROP_BRIGHTNESS, 40)
cap.set(cv2.CAP_PROP_CONTRAST,   40)

# Color ranges
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'blue':   ([100, 43,  46],  [124, 255, 255]),
    'yellow': ([26,  43,  46],  [34,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

# Setup motors
motors.setup()

# Warm up camera
print('Warming up camera...')
for _ in range(20):
    cap.read()
    time.sleep(0.05)

ret, test_frame = cap.read()
if ret:
    print('Camera ready!')
else:
    print('ERROR: Could not open camera.')
print('Motors ready!')
print()
print('>>> Make sure the robot is on the floor with space to move before running the follow cells.')

Warming up camera...
Camera ready!
Motors ready!

>>> Make sure the robot is on the floor with space to move before running the follow cells.


---

## YOUR TURN -- Tweak Zone 1: Configure Your Follow Behavior

These are the key parameters that control how your robot follows the color target. Read each one carefully before running.

- **TARGET_COLOR** -- which color to follow
- **FOLLOW_SPEED** -- how fast the robot drives forward when centered on the target
- **TURN_SPEED** -- how fast the robot turns when the target is off center
- **DEAD_ZONE** -- how many pixels off center before the robot starts turning
- **MIN_RADIUS** -- minimum blob size to trigger following

> **Think like an engineer:** What happens if DEAD_ZONE is too small? What happens if it is too large? What real ADAS system uses a dead zone concept?

In [2]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
TARGET_COLOR = 'red'    # your team color
FOLLOW_SPEED = 35       # forward speed (0-100) -- start low!
TURN_SPEED   = 30       # turning speed (0-100)
DEAD_ZONE    = 50       # pixels -- how far off center before turning
MIN_RADIUS   = 20       # minimum blob size to follow
# ═══════════════════════════════════════

color_lower = np.array(COLOR_RANGES[TARGET_COLOR][0])
color_upper = np.array(COLOR_RANGES[TARGET_COLOR][1])
FRAME_CENTER = 320

print(f'Configuration set:')
print(f'  Target color: {TARGET_COLOR}')
print(f'  Follow speed: {FOLLOW_SPEED}')
print(f'  Turn speed:   {TURN_SPEED}')
print(f'  Dead zone:    center +/- {DEAD_ZONE}px ({FRAME_CENTER-DEAD_ZONE} to {FRAME_CENTER+DEAD_ZONE})')
print(f'  Min radius:   {MIN_RADIUS}px')

Configuration set:
  Target color: red
  Follow speed: 35
  Turn speed:   30
  Dead zone:    center +/- 50px (270 to 370)
  Min radius:   20px


---

## YOUR TURN -- Tweak Zone 2: Start Following

Place the robot on the floor. Hold your colored ball in front of the camera.

Run the START cell. The robot will follow your color. Move the ball left and right and watch the robot track it.

Run the STOP cell to stop the robot.

> **Safety:** Keep your hand on the robot or stay close. Start with FOLLOW_SPEED=35 and TURN_SPEED=30 until you are comfortable with the behavior.

> **Think like an engineer:** Watch the robot's behavior carefully. Does it overshoot when turning? Does it drive too fast and miss the target? These are the same challenges real autonomous vehicle engineers face every day.

In [3]:
# START CELL
feed_widget = widgets.Image(format='jpeg', width=640, height=480)
status_widget = widgets.Label(value='Status: Starting...')
display(status_widget, feed_widget)

following = True

def follow_loop():
    while following:
        ret, frame = cap.read()
        if not ret:
            break
        
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, color_lower, color_upper)
        mask = cv2.erode(mask,  None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)
        mask = cv2.GaussianBlur(mask, (3,3), 0)
        cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        
        # draw center line and dead zone
        cv2.line(frame, (FRAME_CENTER, 0), (FRAME_CENTER, 480), (255,255,255), 1)
        cv2.line(frame, (FRAME_CENTER-DEAD_ZONE, 0), (FRAME_CENTER-DEAD_ZONE, 480), (0,255,0), 1)
        cv2.line(frame, (FRAME_CENTER+DEAD_ZONE, 0), (FRAME_CENTER+DEAD_ZONE, 480), (0,255,0), 1)
        
        if len(cnts) > 0:
            cnt = max(cnts, key=cv2.contourArea)
            (cx, cy), radius = cv2.minEnclosingCircle(cnt)
            
            if radius > MIN_RADIUS:
                cv2.circle(frame, (int(cx), int(cy)), int(radius), (0,255,255), 2)
                cv2.circle(frame, (int(cx), int(cy)), 5, (0,255,255), -1)
                
                # Decision logic
                if cx < FRAME_CENTER - DEAD_ZONE:
                    motors.turn_left(TURN_SPEED)
                    action = 'TURNING LEFT'
                elif cx > FRAME_CENTER + DEAD_ZONE:
                    motors.turn_right(TURN_SPEED)
                    action = 'TURNING RIGHT'
                else:
                    motors.forward(FOLLOW_SPEED)
                    action = 'FORWARD'
                
                cv2.putText(frame, f'{action}  X:{int(cx)}',
                            (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
                status_widget.value = f'Status: {action} -- target at X={int(cx)}, radius={int(radius)}px'
            else:
                motors.brake()
                status_widget.value = 'Status: Target too small -- move closer'
        else:
            motors.brake()
            cv2.putText(frame, 'SEARCHING...', (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            status_widget.value = 'Status: No target detected -- searching'
        
        feed_widget.value = bgr8_to_jpeg(frame)
        time.sleep(0.033)
    
    motors.brake()

follow_thread = threading.Thread(target=follow_loop)
follow_thread.daemon = True
follow_thread.start()
print('Following started. Run the STOP cell to stop the robot.')

Label(value='Status: Starting...')

Image(value=b'', format='jpeg', height='480', width='640')

Following started. Run the STOP cell to stop the robot.


In [4]:
# STOP CELL -- run this to stop the robot
following = False
time.sleep(0.5)
motors.brake()
print('Robot stopped.')

Robot stopped.


---

## YOUR TURN -- Tweak Zone 3: Tune Your Driver

Go back to Tweak Zone 1 and experiment with different values. Try each change one at a time so you can see the effect clearly.

| Change | Expected Effect |
|--------|-----------------|
| Increase FOLLOW_SPEED | Robot drives faster toward target |
| Increase TURN_SPEED | Robot turns faster -- may overshoot |
| Increase DEAD_ZONE | Robot drives straight more, turns less |
| Decrease DEAD_ZONE | Robot turns more aggressively |
| Increase MIN_RADIUS | Only follows large/close targets |
| Decrease MIN_RADIUS | Follows smaller/farther targets |

> **Team challenge:** Find the combination of values that follows the target most smoothly without overshooting. This is your starting configuration for race day.

---

## What Happened?

Think about these questions with your team:

1. What happened when you moved the target very fast? What does that tell you about the robot's reaction time?
2. What happened when you put the target very close vs very far away?
3. The decision logic is very simple -- left, right, or forward. How could you make it smarter?
4. A Tesla following a car in front uses the same basic concept. What extra information would a real car need that our robot doesn't have?

---

## CHALLENGE -- Advanced Students

The current logic has only three states: turn left, turn right, forward. This causes jerky movement.

Implement **proportional control** -- the further the target is from center, the faster the robot turns. A target just outside the dead zone should cause a gentle turn. A target at the edge of the frame should cause a sharp turn.

Hint: calculate the error (distance from center) and multiply it by a gain factor to get the turn speed.

In [ ]:
# YOUR CODE HERE
# ═══════════════════════════════════════
GAIN = 0.1    # how aggressively to respond to error
MAX_TURN_SPEED = 60
MIN_TURN_SPEED = 20
# ═══════════════════════════════════════

def proportional_turn(cx, frame_center, gain=GAIN):
    # Calculate error
    # Calculate turn speed proportional to error
    # Return turn speed clamped between MIN and MAX
    pass


---

## Always clean up when you are done!

In [5]:
following = False
time.sleep(0.5)
motors.cleanup()
cap.release()
print('Motors and camera released.')

Motors and camera released.
